# Adam and Modern Adaptive Optimizers

- Module: 01 Optimization
- Topic: adaptive first-order methods
- Estimated runtime: 10 minutes
- Prerequisites: gradient descent, momentum, coordinate-wise scaling, exponential moving averages
- Expected outputs: adaptive optimizer trajectories on an ill-scaled surface, convergence comparisons, and a high-learning-rate Adam failure experiment

## Learning goals

- Compare AdaGrad, RMSProp, and Adam on the same ill-scaled objective.
- See why coordinate-wise adaptation helps when curvature differs strongly across directions.
- Observe that Adam is still sensitive to learning-rate choices despite its adaptive normalization.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

repo_root = Path.cwd().parents[2]
src_dir = repo_root / "modules" / "01-optimization" / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from optim_utils import (
    adagrad,
    adam,
    gradient_descent,
    make_quadratic,
    make_shifted_least_squares,
    momentum_descent,
    newton_method,
    plot_convergence,
    plot_trajectories,
    rmsprop,
    rosenbrock_objective,
    saddle_objective,
    stochastic_gradient_descent,
    summarize_history,
)

np.set_printoptions(precision=4, suppress=True)


## Ill-scaled quadratic with mismatched curvature

Adaptive methods are easiest to motivate when one coordinate is much steeper than the other. A single global learning rate forces plain gradient descent to choose between safety on the steep axis and speed on the shallow axis.

In [ ]:
objective = make_quadratic(
    matrix=np.array([[40.0, 0.0], [0.0, 0.25]], dtype=float),
    center=np.array([0.0, 0.0], dtype=float),
    name="ill_scaled_quadratic",
)
start = np.array([2.4, 2.0], dtype=float)

gd_history = gradient_descent(objective, start=start, learning_rate=0.045, steps=35)
gd_history["name"] = "GD"
adagrad_history = adagrad(objective, start=start, learning_rate=0.8, steps=35)
adagrad_history["name"] = "AdaGrad"
rmsprop_history = rmsprop(objective, start=start, learning_rate=0.18, steps=35, decay=0.9)
rmsprop_history["name"] = "RMSProp"
adam_history = adam(objective, start=start, learning_rate=0.22, steps=35)
adam_history["name"] = "Adam"
[summarize_history(history, objective) | {"name": history["name"]} for history in [gd_history, adagrad_history, rmsprop_history, adam_history]]

## Trajectory comparison on the same contours

The adaptive methods shrink effective step sizes differently in each coordinate. On this objective, that usually means they stop bouncing in the steep $x_1$ direction while still making visible progress in the flat $x_2$ direction.

In [ ]:
fig, _ = plot_trajectories(
    objective,
    [gd_history, adagrad_history, rmsprop_history, adam_history],
    x_range=(-0.5, 2.8),
    y_range=(-0.5, 2.5),
    title="Adaptive methods on an ill-scaled quadratic",
)
plt.show()
plt.close(fig)

## Objective and gradient-norm comparison

AdaGrad keeps accumulating squared gradients, so its effective learning rate decays over time. RMSProp and Adam use exponential moving averages instead, which often keeps later progress from stalling as quickly.

In [ ]:
fig, _ = plot_convergence(
    [gd_history, adagrad_history, rmsprop_history, adam_history],
    title="Adaptive optimizer comparison",
)
plt.show()
plt.close(fig)

## What happens when Adam's learning rate is too large?

Bias correction and adaptive scaling help, but they do not remove the need for a reasonable global step size. With an overly large learning rate, Adam can still overshoot and produce erratic progress.

In [ ]:
adam_failure = adam(objective, start=start, learning_rate=0.75, steps=20)
adam_failure["name"] = "Adam, lr=0.75"
fig, _ = plot_trajectories(
    objective,
    [adam_history, adam_failure],
    x_range=(-1.0, 3.0),
    y_range=(-1.0, 2.5),
    title="Adam stability depends on the global learning rate",
)
plt.show()
plt.close(fig)

fig, _ = plot_convergence([adam_history, adam_failure], title="Adam failure mode")
plt.show()
plt.close(fig)

{
    "stable_adam_final_value": summarize_history(adam_history, objective)["final_value"],
    "failure_adam_final_value": summarize_history(adam_failure, objective)["final_value"],
}

## Summary

Adaptive optimizers help when different coordinates demand different effective step sizes, but they are not magic. Their internal normalization changes the geometry of the update, yet the global learning rate and moment parameters still control whether the run is smooth or unstable.

## Next steps

- Change the diagonal Hessian entries and see when plain GD becomes competitive again.
- Compare Adam with Nesterov from the previous notebook on the same ill-scaled surface.
- Add a sparse-gradient objective to test AdaGrad's cumulative-memory advantage.